# PSP2 | Planejamento de abastecimento de frotas
Universidade de Brasília • Engenharia de Produção • Prof. André Luiz Marques Serrano

**Fonte:** ANP/LPC, diesel S10, DF e GO, janeiro–junho/2026.

Este notebook contém o pipeline completo e persiste os dados em Volume e tabelas Delta. Executar em Databricks com Python e Spark. Não registra execução bem-sucedida antes de verificar as tabelas.

A análise avalia referências históricas para orçamento e cotação. Não representa economia realizada. Objetivo e perguntas estão no repositório: https://github.com/RafaelLeivas/psp2-planejamento-abastecimento

## 1. Preparar armazenamento persistente
O notebook cria apenas o schema `psp2_abastecimento` no catálogo atual e um volume `arquivos`. As tabelas desse projeto são substituídas em reexecuções. Nenhum outro schema é modificado. Se seu catálogo atual for `hive_metastore` ou `system`, informe um catálogo Unity Catalog ao editar CATALOGO.

In [ ]:
from pathlib import Path
import re, json
CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "psp2_abastecimento"
assert re.fullmatch(r"[A-Za-z0-9_]+", CATALOGO), "Informe um catálogo válido."
assert CATALOGO not in {"hive_metastore", "system"}, "Use um catálogo Unity Catalog, como workspace."
PREFIXO = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PREFIXO}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {PREFIXO}.arquivos")
VOLUME = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/arquivos")
ZIP = VOLUME / "ca-2026-01.zip"
SAIDA = VOLUME / "resultados"
print("Volume persistente:", VOLUME)
print("Arquivo esperado:", ZIP)


## 2. Código do pipeline
Código idêntico ao módulo `scripts/pipeline.py`. Se a origem estiver indisponível, o processamento falha de forma explícita. O recorte não mistura produtos ou unidades.

In [ ]:
"""PSP2: pipeline ANP reproduzível. Python 3.10+, pandas 2.x/3.x.

Uso: python scripts/pipeline.py --zip caminho/ca-2026-01.zip --out resultados
O download oficial é realizado quando --zip não aponta para arquivo existente.
"""
from pathlib import Path
import argparse
import hashlib
import io
import json
import urllib.request
import zipfile
from datetime import datetime, timezone
import pandas as pd

SOURCE = 'https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca/ca-2026-01.zip'
COLS = ['regiao','uf','municipio','revenda','cnpj','logradouro','numero','complemento','bairro','cep','produto','data_coleta','preco_venda','preco_compra','unidade','bandeira']

def stable_id(s):
    return hashlib.sha256(str(s).encode()).hexdigest()[:20]

def run(zip_path, out):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    zip_path = Path(zip_path)
    if not zip_path.exists():
        zip_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(SOURCE, zip_path)
    blob = zip_path.read_bytes()
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        member = next(n for n in archive.namelist() if n.lower().endswith('.csv'))
        raw = pd.read_csv(archive.open(member), sep=';', dtype=str, encoding='utf-8-sig')
    original_columns = list(raw.columns)
    assert len(raw.columns) == len(COLS), 'Esquema da fonte mudou; revisar mapeamento.'
    raw.columns = COLS
    for c in COLS:
        raw[c] = raw[c].str.strip().replace('', pd.NA)
    scope = raw.loc[raw.uf.isin(['DF','GO']) & raw.produto.eq('DIESEL S10')].copy()
    assert not scope.empty, 'Recorte vazio.'
    scope.to_csv(out/'bronze_recorte.csv',index=False)
    profile = []
    for c in COLS:
        s = scope[c]
        profile.append({'atributo':c,'linhas':len(s),'nulos':int(s.isna().sum()),'pct_nulos':round(s.isna().mean()*100,4),'distintos':int(s.nunique()),'exemplos':' | '.join(s.dropna().unique()[:8])})
    pd.DataFrame(profile).to_csv(out/'qualidade_atributos.csv',index=False)
    n_exact = int(scope.duplicated().sum())
    df = scope.drop_duplicates().copy()
    df['data_coleta'] = pd.to_datetime(df.data_coleta,format='%d/%m/%Y',errors='coerce')
    df['preco_venda'] = pd.to_numeric(df.preco_venda.str.replace(',','.',regex=False),errors='coerce')
    df['cnpj'] = df.cnpj.str.replace(r'\D','',regex=True)
    checks = pd.DataFrame({
        'data_invalida':df.data_coleta.isna() | ~df.data_coleta.between('2026-01-01','2026-06-30'),
        'preco_invalido':df.preco_venda.isna() | df.preco_venda.le(0),
        'cnpj_invalido':~df.cnpj.str.fullmatch(r'\d{14}',na=False),
        'localidade_ausente':df.municipio.isna(),
        'unidade_invalida':~df.unidade.eq('R$ / litro'),
        'regiao_inconsistente':~df.regiao.eq('CO'),
    })
    invalid = checks.any(axis=1)
    quarantine = df.loc[invalid].copy()
    quarantine['motivos'] = [';'.join(checks.columns[row.to_numpy(dtype=bool)]) for _,row in checks.loc[invalid].iterrows()]
    quarantine.to_csv(out/'quarentena.csv',index=False)
    df = df.loc[~invalid].copy()
    keys = ['cnpj','produto','data_coleta']
    conflict = df.duplicated(keys, keep=False)
    df.loc[conflict].to_csv(out/'conflitos_chave.csv',index=False)
    assert not conflict.any(), 'Chaves conflitantes: resolver antes de analisar; arquivo conflitos_chave.csv.'
    assert not df.empty
    df['semana'] = df.data_coleta - pd.to_timedelta(df.data_coleta.dt.weekday,unit='D')
    df['localidade_id'] = (df.uf+'|'+df.municipio).map(stable_id)
    # Versão de posto inclui atributos descritivos; preserva mudanças históricas sem escolher arbitrariamente o último nome.
    df['posto_id'] = df[['cnpj','revenda','bandeira','logradouro','numero','bairro','cep']].fillna('').agg('|'.join,axis=1).map(stable_id)
    df['data_id'] = df.data_coleta.dt.strftime('%Y%m%d').astype(int)
    df['produto_id'] = 'diesel_s10'
    df['observacao_id'] = df[keys].astype(str).agg('|'.join,axis=1).map(stable_id)
    dates = df[['data_id','data_coleta','semana']].drop_duplicates().sort_values('data_id')
    dates['mes'] = dates.data_coleta.dt.month
    loc = df[['localidade_id','uf','municipio','regiao']].drop_duplicates()
    posts = df[['posto_id','cnpj','revenda','bandeira','logradouro','numero','bairro','cep']].drop_duplicates()
    prod = pd.DataFrame([{'produto_id':'diesel_s10','produto':'DIESEL S10','unidade':'R$ / litro'}])
    fact = df[['observacao_id','data_id','localidade_id','posto_id','produto_id','preco_venda']]
    tables = {'dim_data':dates,'dim_localidade':loc,'dim_posto':posts,'dim_produto':prod,'fato_preco':fact}
    for name, table in tables.items(): table.to_csv(out/(name+'.csv'),index=False)
    # Um voto por posto por semana; múltiplas coletas do mesmo posto são resumidas pela mediana.
    pw = df.groupby(['uf','municipio','localidade_id','semana','cnpj'],as_index=False).agg(preco=('preco_venda','median'),n_coletas=('preco_venda','size'))
    group = ['uf','municipio','localidade_id','semana']
    weekly = pw.groupby(group).agg(n_postos=('cnpj','nunique'),media=('preco','mean'),mediana=('preco','median'),minimo=('preco','min'),maximo=('preco','max'),desvio=('preco','std')).reset_index()
    qs = pw.groupby(group).preco.quantile([.25,.75]).unstack().reset_index().rename(columns={.25:'p25',.75:'p75'})
    weekly = weekly.merge(qs,on=group,validate='one_to_one')
    weekly['elegivel'] = weekly.n_postos.ge(5)
    weekly['semana_completa'] = weekly.semana.between('2026-01-05','2026-06-22')
    weekly['analise'] = weekly.elegivel & weekly.semana_completa
    weekly['iqr'] = weekly.p75-weekly.p25
    weekly['gap_litro'] = weekly.mediana-weekly.p25
    weekly['gap_pct'] = 100*weekly.gap_litro/weekly.mediana
    weekly['cv_pct'] = 100*weekly.desvio/weekly.media
    weekly.to_csv(out/'indicadores_semanais.csv',index=False)
    valid = weekly.loc[weekly.analise].copy()
    assert not valid.empty
    summary = valid.groupby(['uf','municipio'],as_index=False).agg(semanas=('semana','nunique'),postos_medios=('n_postos','mean'),mediana_media=('mediana','mean'),iqr_medio=('iqr','mean'),gap_medio=('gap_litro','mean'),gap_pct_medio=('gap_pct','mean'),fracao_semanas_gap3=('gap_pct',lambda s:float(s.ge(3).mean())))
    summary['hipotese_atendida'] = (summary.fracao_semanas_gap3.ge(.5) & summary.semanas.ge(12))
    summary['cobertura_suficiente'] = summary.semanas.ge(12)
    summary.to_csv(out/'resumo_localidades.csv',index=False)
    ref = valid[group+['mediana']]
    ranked = pw.merge(ref,on=group,how='inner',validate='many_to_one')
    ranked['abaixo'] = ranked.preco.lt(ranked.mediana)
    ranked['desconto_pct'] = 100*(ranked.mediana-ranked.preco)/ranked.mediana
    ranks = ranked.groupby(['uf','municipio','cnpj'],as_index=False).agg(semanas_observadas=('semana','nunique'),fracao_abaixo=('abaixo','mean'),desconto_medio_pct=('desconto_pct','mean'))
    available = valid.groupby(['uf','municipio']).semana.nunique().rename('semanas_localidade').reset_index()
    ranks = ranks.merge(available,on=['uf','municipio'],validate='many_to_one')
    ranks['cobertura'] = ranks.semanas_observadas/ranks.semanas_localidade
    ranks['elegivel'] = ranks.semanas_observadas.ge(12) & ranks.cobertura.ge(.75)
    ranks = ranks.sort_values(['elegivel','fracao_abaixo','desconto_medio_pct'],ascending=False)
    ranks.to_csv(out/'ranking_postos.csv',index=False)
    scenarios = pd.concat([valid.assign(litros_semanais=v,custo_referencia=valid.mediana*v,custo_cenario=valid.p25*v,economia_potencial=valid.gap_litro*v) for v in [1000,5000,10000]],ignore_index=True)
    scenarios.to_csv(out/'cenarios.csv',index=False)
    # Sinalização de extremos: não excluir preços válidos somente por serem extremos.
    unusual = int((df.preco_venda.lt(2) | df.preco_venda.gt(15)).sum())
    audit = {'source_url':SOURCE,'arquivo_interno':member,'sha256_zip':hashlib.sha256(blob).hexdigest(),'bytes_zip':len(blob),'executado_em_utc':datetime.now(timezone.utc).isoformat(),'ambiente':'local_python','linhas_fonte':len(raw),'linhas_recorte':len(scope),'duplicatas_exatas_removidas':n_exact,'linhas_quarentena':int(invalid.sum()),'linhas_finais':len(df),'problemas':{c:int(checks[c].sum()) for c in checks},'precos_fora_faixa_alerta_2_15':unusual,'data_min':str(df.data_coleta.min().date()),'data_max':str(df.data_coleta.max().date()),'postos':int(df.cnpj.nunique()),'localidades':len(loc),'semanas_elegiveis':len(valid),'colunas_originais':original_columns}
    (out/'auditoria.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(audit,ensure_ascii=False,indent=2))
    return tables, weekly, summary, ranks, scenarios, audit



## 3. Coleta e processamento
A Free Edition pode bloquear acesso à ANP. Se o download falhar, baixe o ZIP diretamente da URL oficial e use **Catalog → catálogo exibido acima → psp2_abastecimento → arquivos → Upload**. Mantenha o nome `ca-2026-01.zip` e execute esta célula novamente. Não é necessário criar token.

Origem: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca/ca-2026-01.zip

In [ ]:
try:
    tables, weekly, summary, ranks, scenarios, audit = run(ZIP, SAIDA)
except Exception as exc:
    print("Se houver bloqueio de rede, envie ca-2026-01.zip ao volume", VOLUME)
    raise
audit["ambiente"] = "databricks"
audit["volume"] = str(VOLUME)
(SAIDA / "auditoria.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.DataFrame([{k:v for k,v in audit.items() if not isinstance(v,(dict,list))}]))


## 4. Persistir Bronze, modelo estrela e resultados em Delta
O grão da fato é CNPJ/produto/data. A dimensão posto preserva versões descritivas. IDs usam hash determinístico, com validação de unicidade e chaves estrangeiras.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType, TimestampType
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype, is_datetime64_any_dtype
def persistir(nome, pdf):
    campos=[]
    for col in pdf.columns:
        dtype=pdf[col].dtype
        typ = (BooleanType() if is_bool_dtype(dtype) else LongType() if is_integer_dtype(dtype)
               else DoubleType() if is_numeric_dtype(dtype) else TimestampType() if is_datetime64_any_dtype(dtype)
               else StringType())
        campos.append(StructField(col,typ,True))
    registros=[]
    for linha in pdf.itertuples(index=False,name=None):
        vals=[]
        for value, campo in zip(linha,campos):
            if pd.isna(value): vals.append(None)
            elif isinstance(campo.dataType,TimestampType): vals.append(pd.Timestamp(value).to_pydatetime())
            elif isinstance(campo.dataType,LongType): vals.append(int(value))
            elif isinstance(campo.dataType,DoubleType): vals.append(float(value))
            elif isinstance(campo.dataType,BooleanType): vals.append(bool(value))
            else: vals.append(str(value))
        registros.append(tuple(vals))
    sdf=spark.createDataFrame(registros,StructType(campos))
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{PREFIXO}.{nome}")
    n=spark.table(f"{PREFIXO}.{nome}").count()
    assert n==len(pdf),f"Contagem divergente em {nome}"
    return {"tabela":f"{PREFIXO}.{nome}","linhas":n}
evidencia=[]
bronze=pd.read_csv(SAIDA/"bronze_recorte.csv",dtype=str)
evidencia.append(persistir("bronze_precos",bronze))
for nome,pdf in tables.items(): evidencia.append(persistir(nome,pdf))
for nome,pdf in {"indicadores_semanais":weekly,"resumo_localidades":summary,"ranking_postos":ranks,"cenarios":scenarios,
                 "qualidade_atributos":pd.read_csv(SAIDA/"qualidade_atributos.csv")}.items():
    evidencia.append(persistir(nome,pdf))
for nome,chave in {"dim_data":"data_id","dim_localidade":"localidade_id","dim_posto":"posto_id","dim_produto":"produto_id","fato_preco":"observacao_id"}.items():
    tabela=spark.table(f"{PREFIXO}.{nome}")
    assert tabela.count()==tabela.select(chave).distinct().count(),f"Chave duplicada em {nome}"
f=spark.table(f"{PREFIXO}.fato_preco")
for nome,chave in {"dim_data":"data_id","dim_localidade":"localidade_id","dim_posto":"posto_id","dim_produto":"produto_id"}.items():
    assert f.join(spark.table(f"{PREFIXO}.{nome}"),chave,"left_anti").count()==0, f"FK órfã: {chave}"
display(pd.DataFrame(evidencia))
print("Persistência e integridade verificadas por releitura das tabelas Delta.")


## 5. Qualidade por atributo
Nulo em complemento pode significar endereço sem informação adicional. Preço de compra não é publicado nesta série após agosto de 2020: não imputar zero e não calcular margens comerciais. Regras sintáticas não comprovam acurácia frente ao preço real de balcão.

In [ ]:
display(spark.table(f"{PREFIXO}.qualidade_atributos"))
print(json.dumps(audit["problemas"],ensure_ascii=False,indent=2))

## 6. Perguntas de negócio
Q1: evolução semanal. Q2: dispersão. Q3: frequência de preço inferior à referência local. Q4: cenários de gasto. Q5: regras propostas e limites.

Semanas elegíveis: cinco postos ou mais e semana inteira dentro do semestre. Localidades com menos de 12 semanas ficam inconclusivas para avaliação da hipótese. Ranking de postos exige ≥12 semanas e ≥75% de cobertura.

In [ ]:
display(spark.sql(f"""SELECT uf,municipio,semana,n_postos,mediana,p25,p75,iqr,gap_pct
FROM {PREFIXO}.indicadores_semanais WHERE analise ORDER BY municipio,semana"""))
display(spark.sql(f"""SELECT * FROM {PREFIXO}.resumo_localidades
ORDER BY cobertura_suficiente DESC,iqr_medio DESC"""))
display(spark.sql(f"""SELECT * FROM {PREFIXO}.ranking_postos WHERE elegivel
ORDER BY fracao_abaixo DESC,desconto_medio_pct DESC LIMIT 20"""))
display(spark.sql(f"""SELECT uf,municipio,litros_semanais,COUNT(*) semanas,
AVG(custo_referencia) gasto_referencia_semanal,AVG(economia_potencial) diferenca_semanal_simulada
FROM {PREFIXO}.cenarios GROUP BY uf,municipio,litros_semanais ORDER BY uf,municipio,litros_semanais"""))


## 7. Gráficos e interpretação
Os resultados abaixo devem ser comparados com a análise local. A hipótese não deve ser reescrita para favorecer o resultado.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(11,5))
for city in ["BRASILIA","GOIANIA","ANAPOLIS"]:
    part=weekly.loc[weekly.municipio.eq(city)&weekly.analise].sort_values("semana")
    ax.plot(part.semana,part.mediana,label=city.title())
ax.set(title="Diesel S10 | mediana semanal",ylabel="R$/litro",xlabel="Semana de coleta")
ax.legend();ax.grid(alpha=.2);fig.tight_layout();display(fig);plt.close(fig)
eligible=summary.loc[summary.cobertura_suficiente]
print("Localidades com cobertura suficiente:",len(eligible))
print("Localidades que atendem ao critério da hipótese:",int(eligible.hipotese_atendida.sum()))
print("Regra proposta: cotação local semanal; monitorar desvio do orçamento; verificar disponibilidade e custo de deslocamento antes de contratar.")


## 8. Evidência persistida e execução independente
Execute a célula final em uma nova sessão depois de concluir o pipeline. Ela consulta somente tabelas e arquivos persistentes. Capture: tabela de contagens, DESCRIBE DETAIL e resultados das perguntas. O notebook sozinho, sem execução, não comprova persistência.

In [ ]:
from datetime import datetime,timezone
manifesto={"executado_em_utc":datetime.now(timezone.utc).isoformat(),"ambiente":"databricks","prefixo":PREFIXO,
           "volume":str(VOLUME),"sha256_zip":audit["sha256_zip"],"tabelas":evidencia,"validacoes":"contagens, PK e FK aprovadas"}
(VOLUME/"evidencia_execucao.json").write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding="utf-8")
display(spark.sql(f"DESCRIBE DETAIL {PREFIXO}.fato_preco"))
print("Registro persistente:",VOLUME/"evidencia_execucao.json")


In [ ]:
# Célula independente: pode ser executada após reiniciar a sessão.
catalogo=spark.sql("SELECT current_catalog()").first()[0]
display(spark.sql(f"SELECT COUNT(*) observacoes,MIN(preco_venda) menor_preco,MAX(preco_venda) maior_preco FROM {catalogo}.psp2_abastecimento.fato_preco"))
display(spark.sql(f"SHOW TABLES IN {catalogo}.psp2_abastecimento"))
